# Declarative Multi-Step LLM Workflow Optimisation
### From Imperative Prompt Chains to Optimisable Programs

**NLP Portfolio -- DSPy-Style Declarative AI Programming**

---

Every production LLM application is a **multi-step workflow**: extract,
classify, reason, summarise, verify. Most teams build these as brittle
chains of hand-written prompts. When the LLM changes, everything breaks.

DSPy introduces a radical alternative: **declare what each step should do**,
then let an optimiser figure out the prompts, demonstrations, and
instructions automatically.

### What This Notebook Covers

| Part | Topic | Key Insight |
|---|---|---|
| 1 | The Problem with Prompt Chains | Why hand-written prompts are fragile |
| 2 | The Declarative Abstraction | Signatures, Modules, Teleprompters |
| 3 | Build a 4-Step Workflow | Extract -> Classify -> Reason -> Summarise |
| 4 | Imperative Baseline | Hand-written prompts for each step |
| 5 | Declarative Pipeline | Same workflow as composable modules |
| 6 | Automatic Optimisation | BootstrapFewShot + instruction tuning |
| 7 | Full Comparison | 5 approaches, 8 visualisations |
| 8 | Production Patterns | Real DSPy code you can deploy |

## The Core Problem: Prompt Engineering Doesn't Scale

```
         IMPERATIVE (today)              DECLARATIVE (DSPy)
         ─────────────────              ──────────────────

Step 1:  'Extract the key entities      Signature:
          from the following text.        text -> entities
          Return JSON with...'

Step 2:  'Classify the topic of the     Signature:
          following text into one of      text, entities -> topic
          these categories: ...'

Step 3:  'Given these entities and       Signature:
          this topic, explain the         entities, topic -> analysis
          main argument using...'

Step 4:  'Summarise the analysis in      Signature:
          exactly 2 sentences. Be         analysis -> summary
          concise and ...'
```

**The imperative version** requires rewriting every prompt when:
- You switch from GPT-4 to Llama 3
- The domain changes from news to medical texts
- Output format requirements change

**The declarative version** only changes the optimiser's training data.

## Environment Setup

In [ ]:
import subprocess, sys
for pkg in ['pandas', 'numpy', 'matplotlib', 'seaborn', 'plotly']:
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
print('Ready.')

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import re, math, copy, hashlib, textwrap, time
from collections import Counter, defaultdict
from dataclasses import dataclass, field
from typing import Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

pd.set_option('display.max_columns', 20)
pd.set_option('display.max_colwidth', 120)
plt.rcParams['figure.figsize'] = (14, 5)
sns.set_style('whitegrid')
np.random.seed(42)
print('Imports OK')

## Dataset: Technical Article Analysis

40 technical articles spanning 5 domains. Each article requires a
4-step analysis pipeline: entity extraction, topic classification,
reasoning / argument analysis, and concise summarisation.

Ground truth was created by experts for evaluation.

In [ ]:
ARTICLES = [
    # AI / Deep Learning (8)
    {'id': 'ai01', 'text': 'Transformers have revolutionised NLP by replacing recurrence with self-attention. '
     'Multi-head attention allows the model to focus on different relationships simultaneously. '
     'The key innovation is computing attention weights as softmax(QK^T/sqrt(d_k))V. '
     'This parallel computation makes transformers much faster to train than RNNs.',
     'gt_entities': ['Transformers', 'self-attention', 'multi-head attention', 'RNNs'],
     'gt_topic': 'deep_learning', 'gt_domain': 'ai'},
    {'id': 'ai02', 'text': 'Gradient vanishing remains a challenge in deep networks. '
     'ResNet introduced skip connections that allow gradients to flow directly through the network. '
     'Batch normalisation further stabilises training by normalising layer activations. '
     'Together, these techniques enable training of networks with hundreds of layers.',
     'gt_entities': ['gradient vanishing', 'ResNet', 'skip connections', 'batch normalisation'],
     'gt_topic': 'deep_learning', 'gt_domain': 'ai'},
    {'id': 'ai03', 'text': 'Large language models like GPT-4 learn through next-token prediction on massive corpora. '
     'Instruction tuning aligns the model to follow human directives. '
     'RLHF refines this alignment using human preference data. '
     'Constitutional AI offers a principled alternative by defining behavioural rules.',
     'gt_entities': ['GPT-4', 'instruction tuning', 'RLHF', 'Constitutional AI'],
     'gt_topic': 'llm_alignment', 'gt_domain': 'ai'},
    {'id': 'ai04', 'text': 'Diffusion models generate images by learning to reverse a noising process. '
     'Stable Diffusion operates in latent space using a VAE encoder for efficiency. '
     'Classifier-free guidance balances quality and diversity during sampling. '
     'ControlNet adds spatial conditioning like edge maps and depth.',
     'gt_entities': ['diffusion models', 'Stable Diffusion', 'classifier-free guidance', 'ControlNet'],
     'gt_topic': 'generative_models', 'gt_domain': 'ai'},
    {'id': 'ai05', 'text': 'LoRA enables efficient fine-tuning by training low-rank matrices added to frozen weights. '
     'QLoRA extends this by quantising the base model to 4-bit precision. '
     'Adapters offer an alternative by inserting small trainable layers. '
     'These parameter-efficient methods make fine-tuning accessible on consumer GPUs.',
     'gt_entities': ['LoRA', 'QLoRA', 'adapters', 'parameter-efficient fine-tuning'],
     'gt_topic': 'fine_tuning', 'gt_domain': 'ai'},
    {'id': 'ai06', 'text': 'Knowledge distillation transfers capability from a large teacher to a smaller student model. '
     'The student learns to match the teacher soft probability distributions. '
     'Pruning removes unnecessary weights after training. '
     'Quantisation reduces numerical precision to INT8 or INT4 for edge deployment.',
     'gt_entities': ['knowledge distillation', 'pruning', 'quantisation', 'edge deployment'],
     'gt_topic': 'model_compression', 'gt_domain': 'ai'},
    {'id': 'ai07', 'text': 'Vision Transformers divide images into patches and process them with self-attention. '
     'DINOv2 learns visual features through self-supervised distillation without labels. '
     'SAM (Segment Anything) provides zero-shot segmentation from point prompts. '
     'These foundation models reduce the need for task-specific training.',
     'gt_entities': ['Vision Transformers', 'DINOv2', 'SAM', 'foundation models'],
     'gt_topic': 'computer_vision', 'gt_domain': 'ai'},
    {'id': 'ai08', 'text': 'Retrieval-Augmented Generation combines retrieval with LLM generation for grounded answers. '
     'Dense passage retrieval encodes queries and documents as vectors. '
     'Self-RAG lets the model decide when retrieval is needed. '
     'CRAG evaluates retrieval quality and falls back to web search if needed.',
     'gt_entities': ['RAG', 'dense passage retrieval', 'Self-RAG', 'CRAG'],
     'gt_topic': 'retrieval_augmented', 'gt_domain': 'ai'},

    # Software Engineering (8)
    {'id': 'se01', 'text': 'Microservices decompose applications into independently deployable services. '
     'API gateways handle routing, rate limiting, and authentication. '
     'Service meshes like Istio manage inter-service communication. '
     'Circuit breakers prevent cascade failures when downstream services are unavailable.',
     'gt_entities': ['microservices', 'API gateways', 'Istio', 'circuit breakers'],
     'gt_topic': 'distributed_systems', 'gt_domain': 'software'},
    {'id': 'se02', 'text': 'Kubernetes orchestrates containerised workloads across clusters. '
     'Pods are the smallest deployable unit, containing one or more containers. '
     'Horizontal Pod Autoscaler adjusts replicas based on CPU or custom metrics. '
     'Helm charts package Kubernetes manifests for reproducible deployments.',
     'gt_entities': ['Kubernetes', 'Pods', 'Horizontal Pod Autoscaler', 'Helm'],
     'gt_topic': 'infrastructure', 'gt_domain': 'software'},
    {'id': 'se03', 'text': 'Event-driven architecture uses message queues for asynchronous communication. '
     'Kafka provides durable, ordered, partitioned event streams. '
     'Event sourcing stores all state changes as immutable events. '
     'CQRS separates read and write models for different optimisation needs.',
     'gt_entities': ['event-driven architecture', 'Kafka', 'event sourcing', 'CQRS'],
     'gt_topic': 'distributed_systems', 'gt_domain': 'software'},
    {'id': 'se04', 'text': 'GraphQL lets clients request exactly the data they need in a single query. '
     'Schema-first design ensures API contracts are explicit and versionable. '
     'DataLoader batches and caches database requests to avoid N+1 queries. '
     'Federation allows composing multiple GraphQL services into one graph.',
     'gt_entities': ['GraphQL', 'schema-first design', 'DataLoader', 'federation'],
     'gt_topic': 'api_design', 'gt_domain': 'software'},
    {'id': 'se05', 'text': 'CI/CD pipelines automate testing and deployment for rapid delivery. '
     'GitHub Actions enables workflow automation with YAML configuration. '
     'Feature flags decouple deployment from release, allowing gradual rollouts. '
     'Canary deployments route a fraction of traffic to new versions for validation.',
     'gt_entities': ['CI/CD', 'GitHub Actions', 'feature flags', 'canary deployments'],
     'gt_topic': 'devops', 'gt_domain': 'software'},
    {'id': 'se06', 'text': 'WebAssembly enables near-native performance in web browsers. '
     'Rust compiles to WASM with zero runtime overhead. '
     'WASI extends WebAssembly beyond browsers to server-side and edge environments. '
     'Component Model provides composable, sandboxed modules with defined interfaces.',
     'gt_entities': ['WebAssembly', 'Rust', 'WASI', 'Component Model'],
     'gt_topic': 'web_performance', 'gt_domain': 'software'},
    {'id': 'se07', 'text': 'SQLite is the most deployed database in the world, embedded in every smartphone. '
     'Write-ahead logging enables concurrent readers with a single writer. '
     'FTS5 provides full-text search with BM25 ranking built into SQLite. '
     'Litestream replicates SQLite databases to S3 for disaster recovery.',
     'gt_entities': ['SQLite', 'write-ahead logging', 'FTS5', 'Litestream'],
     'gt_topic': 'databases', 'gt_domain': 'software'},
    {'id': 'se08', 'text': 'Zero-trust security assumes breach and verifies every request. '
     'mTLS encrypts and authenticates all service-to-service communication. '
     'OAuth 2.0 with PKCE secures public client authorisation flows. '
     'OWASP Top 10 catalogues the most critical web application vulnerabilities.',
     'gt_entities': ['zero-trust', 'mTLS', 'OAuth 2.0 PKCE', 'OWASP Top 10'],
     'gt_topic': 'security', 'gt_domain': 'software'},

    # Data Science (8)
    {'id': 'ds01', 'text': 'XGBoost implements gradient boosting with regularisation for tabular data. '
     'Tree pruning via max_depth and min_child_weight controls overfitting. '
     'Feature importance computed from split gains reveals predictive features. '
     'SHAP values provide theoretically grounded, additive feature attributions.',
     'gt_entities': ['XGBoost', 'gradient boosting', 'feature importance', 'SHAP'],
     'gt_topic': 'supervised_learning', 'gt_domain': 'data_science'},
    {'id': 'ds02', 'text': 'Time series forecasting decomposes signals into trend, seasonality, and residuals. '
     'ARIMA models autocorrelation with (p,d,q) parameters. '
     'Prophet handles multiple seasonalities and holiday effects automatically. '
     'Neural forecasters like N-BEATS learn patterns without manual feature engineering.',
     'gt_entities': ['time series', 'ARIMA', 'Prophet', 'N-BEATS'],
     'gt_topic': 'forecasting', 'gt_domain': 'data_science'},
    {'id': 'ds03', 'text': 'Causal inference separates correlation from causation using structural assumptions. '
     'DAGs (directed acyclic graphs) encode causal relationships. '
     'Instrumental variables provide unbiased estimates when confounders exist. '
     'Double machine learning combines ML predictions with econometric identification.',
     'gt_entities': ['causal inference', 'DAGs', 'instrumental variables', 'double ML'],
     'gt_topic': 'causal_analysis', 'gt_domain': 'data_science'},
    {'id': 'ds04', 'text': 'A/B testing compares two variants using statistical hypothesis testing. '
     'Sample size depends on minimum detectable effect and desired power. '
     'Sequential testing allows early stopping without inflating false positive rates. '
     'CUPED reduces variance using pre-experiment covariates for faster convergence.',
     'gt_entities': ['A/B testing', 'sequential testing', 'CUPED', 'statistical power'],
     'gt_topic': 'experimentation', 'gt_domain': 'data_science'},
    {'id': 'ds05', 'text': 'Anomaly detection identifies data points that deviate from expected patterns. '
     'Isolation Forest isolates anomalies by recursive random partitioning. '
     'Autoencoders learn to reconstruct normal data and flag high-error inputs. '
     'DBSCAN clusters dense regions and labels sparse points as noise.',
     'gt_entities': ['anomaly detection', 'Isolation Forest', 'autoencoders', 'DBSCAN'],
     'gt_topic': 'unsupervised_learning', 'gt_domain': 'data_science'},
    {'id': 'ds06', 'text': 'Feature engineering creates predictive representations from raw data. '
     'Target encoding replaces categories with smoothed conditional means. '
     'Polynomial features capture non-linear relationships between variables. '
     'Feature selection via mutual information ranks features by statistical dependence.',
     'gt_entities': ['feature engineering', 'target encoding', 'polynomial features', 'mutual information'],
     'gt_topic': 'feature_eng', 'gt_domain': 'data_science'},
    {'id': 'ds07', 'text': 'Imbalanced classification requires specialised techniques. '
     'SMOTE generates synthetic minority samples by interpolating nearest neighbours. '
     'Focal loss down-weights easy negatives for the classifier to focus on hard cases. '
     'Cost-sensitive learning assigns higher misclassification costs to the minority class.',
     'gt_entities': ['imbalanced classification', 'SMOTE', 'focal loss', 'cost-sensitive learning'],
     'gt_topic': 'supervised_learning', 'gt_domain': 'data_science'},
    {'id': 'ds08', 'text': 'Bayesian optimisation finds the global optimum of expensive black-box functions. '
     'A Gaussian process models the objective function as a probabilistic surrogate. '
     'Acquisition functions like Expected Improvement balance exploration and exploitation. '
     'Optuna implements efficient tree-structured Parzen estimators for hyperparameter search.',
     'gt_entities': ['Bayesian optimisation', 'Gaussian process', 'Expected Improvement', 'Optuna'],
     'gt_topic': 'optimisation', 'gt_domain': 'data_science'},

    # Cloud / MLOps (8)
    {'id': 'cl01', 'text': 'MLflow tracks experiments with parameters, metrics, and artifacts. '
     'Model Registry provides versioning and staging for production models. '
     'MLflow Projects define reproducible environments with conda or docker. '
     'The tracking server persists metadata in a SQL backend with S3 artifact storage.',
     'gt_entities': ['MLflow', 'Model Registry', 'MLflow Projects', 'tracking server'],
     'gt_topic': 'mlops', 'gt_domain': 'cloud'},
    {'id': 'cl02', 'text': 'Feature stores serve precomputed features for training and inference. '
     'Feast provides an open-source feature serving layer with online and offline stores. '
     'Point-in-time joins prevent future data leakage in training datasets. '
     'Feature monitoring detects distribution drift before model quality degrades.',
     'gt_entities': ['feature stores', 'Feast', 'point-in-time joins', 'feature monitoring'],
     'gt_topic': 'mlops', 'gt_domain': 'cloud'},
    {'id': 'cl03', 'text': 'Serverless computing executes functions on-demand without managing servers. '
     'AWS Lambda scales automatically from zero to thousands of instances. '
     'Cold starts add latency when a function has no warm instances ready. '
     'Provisioned concurrency pre-warms instances to eliminate cold start delays.',
     'gt_entities': ['serverless', 'AWS Lambda', 'cold starts', 'provisioned concurrency'],
     'gt_topic': 'cloud_compute', 'gt_domain': 'cloud'},
    {'id': 'cl04', 'text': 'Data lakehouse combines data lake flexibility with data warehouse reliability. '
     'Delta Lake adds ACID transactions and schema enforcement to Parquet files. '
     'Apache Iceberg supports schema evolution and time travel queries. '
     'Medallion architecture organises data into bronze, silver, and gold layers.',
     'gt_entities': ['data lakehouse', 'Delta Lake', 'Apache Iceberg', 'medallion architecture'],
     'gt_topic': 'data_architecture', 'gt_domain': 'cloud'},
    {'id': 'cl05', 'text': 'Model serving deploys ML models as scalable prediction APIs. '
     'TorchServe handles model archiving, batching, and GPU scheduling. '
     'Triton Inference Server supports multi-framework model deployment. '
     'BentoML packages models with pre/post-processing into deployable services.',
     'gt_entities': ['model serving', 'TorchServe', 'Triton', 'BentoML'],
     'gt_topic': 'deployment', 'gt_domain': 'cloud'},
    {'id': 'cl06', 'text': 'Observability combines logs, metrics, and traces for production monitoring. '
     'OpenTelemetry standardises distributed tracing instrumentation. '
     'Prometheus collects time-series metrics with a pull-based model. '
     'Grafana dashboards visualise system health with alerting capabilities.',
     'gt_entities': ['observability', 'OpenTelemetry', 'Prometheus', 'Grafana'],
     'gt_topic': 'monitoring', 'gt_domain': 'cloud'},
    {'id': 'cl07', 'text': 'Infrastructure as Code manages cloud resources through declarative templates. '
     'Terraform uses HCL to define multi-cloud infrastructure. '
     'Pulumi allows IaC in general-purpose languages like Python and TypeScript. '
     'GitOps uses Git as the single source of truth for infrastructure state.',
     'gt_entities': ['Infrastructure as Code', 'Terraform', 'Pulumi', 'GitOps'],
     'gt_topic': 'infrastructure', 'gt_domain': 'cloud'},
    {'id': 'cl08', 'text': 'Vector databases store and search high-dimensional embeddings efficiently. '
     'HNSW indexes enable O(log N) approximate nearest neighbour search. '
     'Product quantisation compresses vectors for memory-efficient storage. '
     'Metadata filtering combines vector similarity with structured attribute queries.',
     'gt_entities': ['vector databases', 'HNSW', 'product quantisation', 'metadata filtering'],
     'gt_topic': 'data_architecture', 'gt_domain': 'cloud'},

    # Research Methods (8)
    {'id': 'rm01', 'text': 'Systematic reviews synthesise evidence across multiple studies. '
     'PRISMA guidelines standardise reporting of search and selection criteria. '
     'Meta-analysis pools effect sizes using fixed or random effects models. '
     'Publication bias is assessed through funnel plots and Egger regression.',
     'gt_entities': ['systematic reviews', 'PRISMA', 'meta-analysis', 'publication bias'],
     'gt_topic': 'research_synthesis', 'gt_domain': 'research'},
    {'id': 'rm02', 'text': 'Randomised controlled trials are the gold standard for causal claims. '
     'Blinding reduces measurement bias by hiding treatment assignment. '
     'Intention-to-treat analysis preserves the randomisation benefit. '
     'Adaptive trial designs modify enrollment based on interim results.',
     'gt_entities': ['RCTs', 'blinding', 'intention-to-treat', 'adaptive designs'],
     'gt_topic': 'experimental_design', 'gt_domain': 'research'},
    {'id': 'rm03', 'text': 'Bayesian statistics updates prior beliefs with observed data via Bayes theorem. '
     'Prior distributions encode domain knowledge before data collection. '
     'MCMC sampling approximates posterior distributions for complex models. '
     'Credible intervals provide direct probability statements about parameters.',
     'gt_entities': ['Bayesian statistics', 'priors', 'MCMC', 'credible intervals'],
     'gt_topic': 'statistical_methods', 'gt_domain': 'research'},
    {'id': 'rm04', 'text': 'Structural equation modelling tests hypothesised causal relationships. '
     'Latent variables represent unobservable constructs through measured indicators. '
     'Fit indices like CFI and RMSEA evaluate how well the model matches data. '
     'Mediation analysis decomposes effects into direct and indirect pathways.',
     'gt_entities': ['SEM', 'latent variables', 'fit indices', 'mediation analysis'],
     'gt_topic': 'statistical_methods', 'gt_domain': 'research'},
    {'id': 'rm05', 'text': 'Grounded theory builds theory inductively from systematically collected data. '
     'Open coding identifies initial concepts in qualitative data. '
     'Theoretical sampling directs data collection based on emerging theory. '
     'Saturation occurs when new data yields no additional theoretical insights.',
     'gt_entities': ['grounded theory', 'open coding', 'theoretical sampling', 'saturation'],
     'gt_topic': 'qualitative_methods', 'gt_domain': 'research'},
    {'id': 'rm06', 'text': 'Reproducibility crisis affects many scientific fields. '
     'Pre-registration prevents hypothesising after results are known. '
     'Open data and code sharing allow independent verification of results. '
     'Registered reports receive peer review before data collection.',
     'gt_entities': ['reproducibility', 'pre-registration', 'open data', 'registered reports'],
     'gt_topic': 'research_integrity', 'gt_domain': 'research'},
    {'id': 'rm07', 'text': 'Power analysis determines the sample size needed to detect a given effect. '
     'Effect size measures the magnitude of a phenomenon (Cohen d, odds ratio). '
     'Type II error (false negative) probability decreases with larger samples. '
     'Post-hoc power analysis is widely discouraged by statisticians.',
     'gt_entities': ['power analysis', 'effect size', 'Type II error', 'sample size'],
     'gt_topic': 'statistical_methods', 'gt_domain': 'research'},
    {'id': 'rm08', 'text': 'Mixed methods research combines quantitative and qualitative approaches. '
     'Sequential designs collect one type of data to inform the other. '
     'Convergent designs collect both simultaneously and compare findings. '
     'Triangulation strengthens validity by combining multiple data sources.',
     'gt_entities': ['mixed methods', 'sequential designs', 'convergent designs', 'triangulation'],
     'gt_topic': 'research_design', 'gt_domain': 'research'},
]

print(f'Dataset: {len(ARTICLES)} articles')
print(f'Domains: {dict(Counter(a["gt_domain"] for a in ARTICLES))}')
print(f'Topics: {len(set(a["gt_topic"] for a in ARTICLES))} unique')

## Simulated LLM Engine

A deterministic LLM simulation that responds to `prompt_quality`,
`chain_of_thought`, and `demonstrations`. This lets us compare
strategies without API costs while preserving realistic behaviour.

In production, swap for `dspy.LM('openai/gpt-4o')` or
`dspy.LM('ollama_chat/qwen3')`.

In [ ]:
class SimulatedLLM:
    STOPWORDS = {'the','a','an','is','are','was','were','be','been','being',
                 'have','has','had','do','does','did','will','would','could',
                 'should','may','might','can','shall','to','of','in','for',
                 'on','with','at','by','from','as','into','about','between',
                 'through','during','and','but','or','nor','not','so','yet',
                 'if','then','it','its','that','this','than','what','which',
                 'how','these','those','their','they','them','each','every',
                 'all','both','more','most','some','any','like','using','used',
                 'provides','without','allows','one','new','no'}

    def __init__(self, noise=0.08):
        self.noise = noise
        self._seed = 42

    def _rng(self):
        self._seed = (self._seed * 6364136223846793005 + 1) & 0xFFFFFFFF
        return (self._seed >> 16) / 65535.0

    def extract_entities(self, text, prompt_quality=0.3, demos=None):
        words = re.findall(r'[A-Z][a-z]+(?:\s+[A-Z][a-z]+)*|[A-Z]{2,}[a-z]*|\b[a-z]+(?:-[a-z]+)+\b', text)
        # Filter out common English words that aren't entities
        entities = [w for w in words if w.lower() not in self.STOPWORDS and len(w) > 2]
        # Dedup preserving order
        seen = set()
        unique = []
        for e in entities:
            el = e.lower()
            if el not in seen:
                seen.add(el)
                unique.append(e)
        # Prompt quality controls how many correct entities we retain
        n_keep = max(2, int(len(unique) * (0.5 + prompt_quality * 0.5)))
        result = unique[:n_keep]
        # Demos boost: add back any missed
        if demos and len(demos) >= 2 and len(result) < len(unique):
            result = unique[:n_keep + 1]
        # Noise: sometimes inject a spurious entity
        if self._rng() < self.noise * (1 - prompt_quality):
            result.append('methodology')
        return result

    def classify_topic(self, text, entities, prompt_quality=0.3,
                       chain_of_thought=False, demos=None):
        # Build a unified text from text + entities
        combined = text.lower() + ' ' + ' '.join(e.lower() for e in entities)

        TOPIC_RULES = [
            ('deep_learning', ['transformer', 'attention', 'resnet', 'gradient',
                               'batch normal', 'cnn', 'rnn', 'convolution', 'skip connection']),
            ('llm_alignment', ['rlhf', 'instruction tuning', 'constitutional', 'alignment',
                               'preference', 'gpt-4']),
            ('generative_models', ['diffusion', 'stable diffusion', 'gan', 'vae',
                                   'controlnet', 'classifier-free']),
            ('fine_tuning', ['lora', 'qlora', 'adapter', 'parameter-efficient', 'fine-tun']),
            ('model_compression', ['distillation', 'pruning', 'quantis', 'edge deploy']),
            ('computer_vision', ['vision transformer', 'dino', 'sam ', 'segment anything',
                                 'image patch']),
            ('retrieval_augmented', ['rag', 'retrieval-augmented', 'dense passage',
                                     'self-rag', 'crag']),
            ('distributed_systems', ['microservice', 'circuit breaker', 'service mesh',
                                     'kafka', 'event sourcing', 'cqrs', 'event-driven']),
            ('infrastructure', ['kubernetes', 'terraform', 'pulumi', 'gitops',
                                'helm', 'pod']),
            ('api_design', ['graphql', 'dataloader', 'federation', 'schema-first']),
            ('devops', ['ci/cd', 'github actions', 'feature flag', 'canary']),
            ('web_performance', ['webassembly', 'wasm', 'wasi', 'rust']),
            ('databases', ['sqlite', 'write-ahead', 'fts5', 'litestream']),
            ('security', ['zero-trust', 'mtls', 'oauth', 'owasp']),
            ('supervised_learning', ['xgboost', 'gradient boosting', 'smote', 'focal loss',
                                     'imbalanced', 'cost-sensitive']),
            ('forecasting', ['time series', 'arima', 'prophet', 'n-beats', 'seasonality']),
            ('causal_analysis', ['causal inference', 'dag', 'instrumental variable',
                                 'double machine learning']),
            ('experimentation', ['a/b test', 'sequential test', 'cuped', 'sample size']),
            ('unsupervised_learning', ['anomaly detection', 'isolation forest', 'dbscan',
                                       'autoencoder']),
            ('feature_eng', ['feature engineering', 'target encoding', 'polynomial',
                             'mutual information']),
            ('optimisation', ['bayesian optimis', 'gaussian process', 'optuna',
                              'acquisition function']),
            ('mlops', ['mlflow', 'feature store', 'feast', 'model registry']),
            ('cloud_compute', ['serverless', 'lambda', 'cold start', 'provisioned']),
            ('data_architecture', ['lakehouse', 'delta lake', 'iceberg', 'medallion',
                                   'vector database', 'hnsw']),
            ('deployment', ['model serving', 'torchserve', 'triton', 'bentoml']),
            ('monitoring', ['observability', 'opentelemetry', 'prometheus', 'grafana']),
            ('research_synthesis', ['systematic review', 'prisma', 'meta-analysis',
                                    'funnel plot']),
            ('experimental_design', ['rct', 'blinding', 'intention-to-treat', 'adaptive']),
            ('statistical_methods', ['bayesian statistics', 'mcmc', 'credible interval',
                                     'sem', 'latent variable', 'power analysis',
                                     'effect size']),
            ('qualitative_methods', ['grounded theory', 'open coding', 'theoretical sampling']),
            ('research_integrity', ['reproducibility', 'pre-registration', 'registered report']),
            ('research_design', ['mixed methods', 'triangulation', 'convergent design']),
        ]

        scores = {}
        for topic, keywords in TOPIC_RULES:
            s = sum(1 for kw in keywords if kw in combined)
            if s > 0:
                scores[topic] = s

        if not scores:
            return 'unknown'

        ranked = sorted(scores.items(), key=lambda x: -x[1])

        # Chain-of-thought: consider second-best and validate
        if chain_of_thought and len(ranked) >= 2:
            top, second = ranked[0], ranked[1]
            if top[1] == second[1]:
                # Tie: CoT picks the more specific one (fewer keywords = more specific)
                for topic, keywords in TOPIC_RULES:
                    if topic == top[0] and len(keywords) < 5:
                        return top[0]
                    if topic == second[0] and len(keywords) < 5:
                        return second[0]

        # Prompt quality: low quality may confuse near-ties
        if prompt_quality < 0.4 and len(ranked) >= 2:
            if ranked[0][1] - ranked[1][1] <= 1 and self._rng() < 0.3:
                return ranked[1][0]  # occasional misclassification

        return ranked[0][0]

    def reason(self, text, entities, topic, prompt_quality=0.3,
               chain_of_thought=False, demos=None):
        sents = re.split(r'(?<=[.!?])\s+', text.strip())
        # Pick the most informative sentences
        scored = []
        ent_tokens = set()
        for e in entities:
            ent_tokens.update(e.lower().split())
        for s in sents:
            sw = set(re.findall(r'[a-z0-9]+', s.lower()))
            score = len(sw & ent_tokens)
            if any(w in s.lower() for w in ['because', 'enables', 'prevents',
                                             'allows', 'combines', 'introduced',
                                             'provides', 'replaces']):
                score += 2
            scored.append((score, s))
        scored.sort(key=lambda x: -x[0])

        n = 1 + int(prompt_quality * 2)
        if chain_of_thought:
            n += 1
        parts = [s for _, s in scored[:n]]

        if chain_of_thought:
            # Add a synthesised reasoning connector
            parts.append(f'These concepts are central to {topic.replace("_", " ")} '
                         f'because they address fundamental challenges in the field.')

        if demos and len(demos) >= 2:
            parts = parts[:3]  # Concise like examples

        return ' '.join(parts)

    def summarise(self, analysis, prompt_quality=0.3,
                  chain_of_thought=False, demos=None):
        sents = re.split(r'(?<=[.!?])\s+', analysis.strip())
        if prompt_quality >= 0.6:
            # Good prompt: pick first + last sentence (intro + conclusion)
            if len(sents) >= 2:
                return sents[0] + ' ' + sents[-1]
            return sents[0]
        elif prompt_quality >= 0.4:
            return sents[0] if sents else analysis
        else:
            # Low quality: truncate raw
            words = analysis.split()
            return ' '.join(words[:25]) + ('.' if len(words) > 25 else '')


LLM = SimulatedLLM(noise=0.08)
print('SimulatedLLM ready (deterministic, no API keys needed)')

## The Declarative Abstraction

### Three Core Concepts

```
1. SIGNATURES -- What each step does
   ┌────────────────────────────────────┐
   │ text -> entities                   │  "Extract technical entities"
   │ text, entities -> topic            │  "Classify the topic"
   │ text, entities, topic -> analysis  │  "Explain the key argument"
   │ analysis -> summary               │  "Summarise in 2 sentences"
   └────────────────────────────────────┘

2. MODULES -- How steps compose
   ┌────────────────────────────────────┐
   │ Predict(sig)        Direct call    │
   │ ChainOfThought(sig) + reasoning   │
   │ MultiChainComparison Select best   │
   └────────────────────────────────────┘

3. TELEPROMPTERS / OPTIMISERS -- How to improve
   ┌────────────────────────────────────┐
   │ BootstrapFewShot  Select demos    │
   │ MIPROv2           + instructions  │
   │ COPRO             Prompt search   │
   └────────────────────────────────────┘
```

### Why This Matters

With imperative prompts, changing one step often breaks the next.
With signatures, each step is **isolated and independently optimisable**.
The optimiser treats the full pipeline as a single program and finds
the best combination of prompts, demonstrations, and instructions.

## Part 3: Building the Multi-Step Workflow

### Step 1: Define Signatures

A Signature is a typed input/output declaration. No prompt text.

```python
# Real DSPy code
class ExtractEntities(dspy.Signature):
    text: str = dspy.InputField(desc='technical article')
    entities: list[str] = dspy.OutputField(desc='key technical entities')

class ClassifyTopic(dspy.Signature):
    text: str = dspy.InputField()
    entities: list[str] = dspy.InputField()
    topic: str = dspy.OutputField(desc='technical topic category')
```

In [ ]:
@dataclass
class Signature:
    name: str
    inputs: list  # field names
    outputs: list  # field names
    desc: str = ''

    def __repr__(self):
        ins = ', '.join(self.inputs)
        outs = ', '.join(self.outputs)
        return f'{self.name}({ins} -> {outs})'


# Define our 4-step signatures
SIG_EXTRACT = Signature('ExtractEntities', ['text'], ['entities'],
                        'Extract key technical entities from the article')
SIG_CLASSIFY = Signature('ClassifyTopic', ['text', 'entities'], ['topic'],
                         'Classify the technical topic from text and entities')
SIG_REASON = Signature('AnalyseArgument', ['text', 'entities', 'topic'], ['analysis'],
                       'Explain the main argument and contribution')
SIG_SUMMARISE = Signature('Summarise', ['analysis'], ['summary'],
                          'Produce a concise 2-sentence summary')

SIGNATURES = [SIG_EXTRACT, SIG_CLASSIFY, SIG_REASON, SIG_SUMMARISE]

print('Workflow Signatures:')
for i, sig in enumerate(SIGNATURES, 1):
    print(f'  Step {i}: {sig}')
    print(f'         {sig.desc}')

### Step 2: Build Modules

Modules wrap signatures with execution strategies:
- **Predict**: call the LLM directly
- **ChainOfThought**: ask the LLM to reason step-by-step before answering
- **Retry**: retry on failure with error feedback

In [ ]:
class Predict:
    def __init__(self, sig, llm, prompt_quality=0.3, demos=None):
        self.sig = sig
        self.llm = llm
        self.pq = prompt_quality
        self.demos = demos

    def __call__(self, **kwargs):
        if self.sig.name == 'ExtractEntities':
            return {'entities': self.llm.extract_entities(
                kwargs['text'], prompt_quality=self.pq, demos=self.demos)}
        elif self.sig.name == 'ClassifyTopic':
            return {'topic': self.llm.classify_topic(
                kwargs['text'], kwargs['entities'],
                prompt_quality=self.pq, demos=self.demos)}
        elif self.sig.name == 'AnalyseArgument':
            return {'analysis': self.llm.reason(
                kwargs['text'], kwargs['entities'], kwargs['topic'],
                prompt_quality=self.pq, demos=self.demos)}
        elif self.sig.name == 'Summarise':
            return {'summary': self.llm.summarise(
                kwargs['analysis'], prompt_quality=self.pq, demos=self.demos)}
        return {}


class ChainOfThought:
    def __init__(self, sig, llm, prompt_quality=0.3, demos=None):
        self.sig = sig
        self.llm = llm
        self.pq = prompt_quality
        self.demos = demos

    def __call__(self, **kwargs):
        if self.sig.name == 'ExtractEntities':
            return {'entities': self.llm.extract_entities(
                kwargs['text'], prompt_quality=self.pq, demos=self.demos)}
        elif self.sig.name == 'ClassifyTopic':
            return {'topic': self.llm.classify_topic(
                kwargs['text'], kwargs['entities'],
                prompt_quality=self.pq, chain_of_thought=True, demos=self.demos)}
        elif self.sig.name == 'AnalyseArgument':
            return {'analysis': self.llm.reason(
                kwargs['text'], kwargs['entities'], kwargs['topic'],
                prompt_quality=self.pq, chain_of_thought=True, demos=self.demos)}
        elif self.sig.name == 'Summarise':
            return {'summary': self.llm.summarise(
                kwargs['analysis'], prompt_quality=self.pq,
                chain_of_thought=True, demos=self.demos)}
        return {}


print('Modules: Predict (direct), ChainOfThought (with reasoning)')

### Step 3: Compose Into a Pipeline

The pipeline chains modules together. Each module's output
feeds into the next module's input -- like PyTorch `nn.Sequential`.

```python
# Real DSPy code
class ArticleAnalyser(dspy.Module):
    def __init__(self):
        self.extract = dspy.Predict(ExtractEntities)
        self.classify = dspy.ChainOfThought(ClassifyTopic)
        self.reason = dspy.ChainOfThought(AnalyseArgument)
        self.summarise = dspy.Predict(Summarise)

    def forward(self, text):
        entities = self.extract(text=text).entities
        topic = self.classify(text=text, entities=entities).topic
        analysis = self.reason(text=text, entities=entities, topic=topic).analysis
        summary = self.summarise(analysis=analysis).summary
        return dspy.Prediction(entities=entities, topic=topic,
                               analysis=analysis, summary=summary)
```

In [ ]:
class MultiStepPipeline:
    def __init__(self, steps, name='Pipeline'):
        self.steps = steps  # list of (sig_name, module)
        self.name = name

    def run(self, text):
        state = {'text': text}
        trace = []
        for sig_name, module in self.steps:
            inputs = {k: v for k, v in state.items() if k in module.sig.inputs}
            output = module(**inputs)
            state.update(output)
            trace.append({'step': sig_name, 'output_keys': list(output.keys())})
        return state, trace


print('MultiStepPipeline: chains modules together, propagating state')

## Evaluation Metrics

Each step has its own metric. The pipeline metric is the **average
across all steps** -- this is essential because optimising one step
might degrade another.

| Step | Metric | What It Measures |
|---|---|---|
| Extract | Entity F1 | Are the right entities found? |
| Classify | Accuracy | Is the topic correct? |
| Reason | Coverage | Does the analysis cover key entities? |
| Summarise | Conciseness | Is the summary short yet faithful? |

In [ ]:
def entity_f1(predicted, ground_truth):
    pred_set = set(e.lower() for e in predicted)
    gt_set = set(e.lower() for e in ground_truth)
    if not pred_set or not gt_set:
        return 0.0
    common = pred_set & gt_set
    # Also check substring matches (e.g. 'resnet' in 'resnet skip connections')
    for p in pred_set:
        for g in gt_set:
            if p in g or g in p:
                common.add(g)
    if not common:
        return 0.0
    p = len(common) / len(pred_set)
    r = len(common) / len(gt_set)
    return 2 * p * r / (p + r) if (p + r) > 0 else 0.0


def classify_accuracy(predicted, ground_truth):
    return 1.0 if predicted == ground_truth else 0.0


def reasoning_coverage(analysis, entities):
    if not entities or not analysis:
        return 0.0
    analysis_lower = analysis.lower()
    covered = sum(1 for e in entities if e.lower() in analysis_lower)
    return covered / len(entities)


def summary_quality(summary, analysis):
    if not summary:
        return 0.0
    # Penalise for being too long
    summary_words = len(summary.split())
    analysis_words = max(len(analysis.split()), 1)
    compression = 1 - (summary_words / analysis_words)
    compression = max(0, min(1, compression))
    # Reward for covering key terms from analysis
    analysis_tokens = set(re.findall(r'[a-z0-9]+', analysis.lower()))
    summary_tokens = set(re.findall(r'[a-z0-9]+', summary.lower()))
    coverage = len(summary_tokens & analysis_tokens) / max(len(analysis_tokens), 1)
    return 0.5 * compression + 0.5 * coverage


def evaluate_pipeline(pipeline, articles):
    results = []
    for art in articles:
        state, trace = pipeline.run(art['text'])

        ef1 = entity_f1(state.get('entities', []), art['gt_entities'])
        acc = classify_accuracy(state.get('topic', ''), art['gt_topic'])
        cov = reasoning_coverage(state.get('analysis', ''), art['gt_entities'])
        sq = summary_quality(state.get('summary', ''), state.get('analysis', ''))
        composite = (ef1 + acc + cov + sq) / 4

        results.append({
            'article_id': art['id'],
            'domain': art['gt_domain'],
            'pipeline': pipeline.name,
            'entity_f1': round(ef1, 4),
            'classify_acc': round(acc, 4),
            'reasoning_cov': round(cov, 4),
            'summary_qual': round(sq, 4),
            'composite': round(composite, 4),
            'topic_pred': state.get('topic', ''),
            'n_entities': len(state.get('entities', [])),
        })
    return pd.DataFrame(results)


print('Metrics defined: Entity F1, Classification Accuracy, '
      'Reasoning Coverage, Summary Quality, Composite Score')

## Part 4: Imperative Baseline

Hand-written prompts. Low prompt quality (0.30) represents a
first-draft attempt -- the kind of prompt most developers start with.

In [ ]:
# Pipeline 1: Imperative -- all Predict, low prompt quality, no demos
pipe_imperative = MultiStepPipeline([
    ('extract', Predict(SIG_EXTRACT, LLM, prompt_quality=0.30)),
    ('classify', Predict(SIG_CLASSIFY, LLM, prompt_quality=0.30)),
    ('reason', Predict(SIG_REASON, LLM, prompt_quality=0.30)),
    ('summarise', Predict(SIG_SUMMARISE, LLM, prompt_quality=0.30)),
], name='Imperative Baseline')

df_imperative = evaluate_pipeline(pipe_imperative, ARTICLES)

print('IMPERATIVE BASELINE')
print('=' * 55)
for col in ['entity_f1', 'classify_acc', 'reasoning_cov', 'summary_qual', 'composite']:
    print(f'  {col:<16s}: {df_imperative[col].mean():.4f}')

## Part 4b: Hand-Tuned Imperative

After hours of prompt engineering: better prompt quality (0.55),
but still no chain-of-thought or demonstrations.

In [ ]:
pipe_tuned = MultiStepPipeline([
    ('extract', Predict(SIG_EXTRACT, LLM, prompt_quality=0.55)),
    ('classify', Predict(SIG_CLASSIFY, LLM, prompt_quality=0.55)),
    ('reason', Predict(SIG_REASON, LLM, prompt_quality=0.55)),
    ('summarise', Predict(SIG_SUMMARISE, LLM, prompt_quality=0.55)),
], name='Hand-Tuned Imperative')

df_tuned = evaluate_pipeline(pipe_tuned, ARTICLES)

print('HAND-TUNED IMPERATIVE')
print('=' * 55)
for col in ['entity_f1', 'classify_acc', 'reasoning_cov', 'summary_qual', 'composite']:
    print(f'  {col:<16s}: {df_tuned[col].mean():.4f}')

## Part 5: Declarative Pipeline with Chain-of-Thought

Same signatures, but now using `ChainOfThought` modules where they
help most: classification and reasoning steps.

In [ ]:
pipe_cot = MultiStepPipeline([
    ('extract', Predict(SIG_EXTRACT, LLM, prompt_quality=0.45)),
    ('classify', ChainOfThought(SIG_CLASSIFY, LLM, prompt_quality=0.45)),
    ('reason', ChainOfThought(SIG_REASON, LLM, prompt_quality=0.45)),
    ('summarise', Predict(SIG_SUMMARISE, LLM, prompt_quality=0.45)),
], name='Declarative + CoT')

df_cot = evaluate_pipeline(pipe_cot, ARTICLES)

print('DECLARATIVE + CoT')
print('=' * 55)
for col in ['entity_f1', 'classify_acc', 'reasoning_cov', 'summary_qual', 'composite']:
    print(f'  {col:<16s}: {df_cot[col].mean():.4f}')

## Part 6: Automatic Optimisation

### The BootstrapFewShot Optimiser

Instead of manually choosing demonstrations, the optimiser:

1. Runs the pipeline on training examples
2. Keeps only the successful examples (where the output matches ground truth)
3. Selects the **most diverse** successful examples as demonstrations
4. Re-evaluates with different prompt quality levels
5. Picks the configuration that maximises the **composite** score

```python
# Real DSPy
optimizer = dspy.BootstrapFewShot(
    metric=composite_metric,
    max_bootstrapped_demos=5,
    max_labeled_demos=3,
)
optimised_pipeline = optimizer.compile(
    ArticleAnalyser(),
    trainset=train_examples,
)
```

In [ ]:
class BootstrapOptimiser:
    def __init__(self, llm, train_articles, val_articles):
        self.llm = llm
        self.train = train_articles
        self.val = val_articles

    def _bootstrap_demos(self, pq=0.5):
        demos = {'extract': [], 'classify': [], 'reason': [], 'summarise': []}
        for art in self.train:
            # Run each step and check if it succeeds
            ents = self.llm.extract_entities(art['text'], prompt_quality=pq)
            ef1 = entity_f1(ents, art['gt_entities'])
            if ef1 > 0.4:
                demos['extract'].append({'text': art['text'][:80], 'entities': ents})

            topic = self.llm.classify_topic(art['text'], ents,
                                           prompt_quality=pq, chain_of_thought=True)
            if topic == art['gt_topic']:
                demos['classify'].append({'text': art['text'][:80],
                                         'entities': ents, 'topic': topic})

            analysis = self.llm.reason(art['text'], ents, art['gt_topic'],
                                      prompt_quality=pq, chain_of_thought=True)
            cov = reasoning_coverage(analysis, art['gt_entities'])
            if cov > 0.5:
                demos['reason'].append({'entities': ents, 'topic': art['gt_topic'],
                                       'analysis': analysis[:100]})

            summary = self.llm.summarise(analysis, prompt_quality=pq)
            sq = summary_quality(summary, analysis)
            if sq > 0.3:
                demos['summarise'].append({'analysis': analysis[:60],
                                          'summary': summary[:60]})

        # Keep top-5 per step
        for k in demos:
            demos[k] = demos[k][:5]
        return demos

    def _build_pipeline(self, module_types, pq, demos):
        steps = []
        configs = [
            ('extract', SIG_EXTRACT, 'extract'),
            ('classify', SIG_CLASSIFY, 'classify'),
            ('reason', SIG_REASON, 'reason'),
            ('summarise', SIG_SUMMARISE, 'summarise'),
        ]
        for step_name, sig, demo_key in configs:
            cls = module_types.get(step_name, Predict)
            d = demos.get(demo_key) if demos else None
            steps.append((step_name, cls(sig, self.llm, prompt_quality=pq, demos=d)))
        return MultiStepPipeline(steps, name='trial')

    def optimise(self):
        print('BootstrapFewShot optimisation...')
        print(f'  Train: {len(self.train)} | Val: {len(self.val)}')

        # Phase 1: Bootstrap demonstrations
        print('  Phase 1: Bootstrapping demonstrations...')
        demos = self._bootstrap_demos(pq=0.5)
        for k, v in demos.items():
            print(f'    {k}: {len(v)} demos bootstrapped')

        # Phase 2: Search over configurations
        print('  Phase 2: Searching configurations...')
        trials = []
        best_score = 0
        best_config = {}

        module_combos = [
            {'extract': Predict, 'classify': Predict,
             'reason': Predict, 'summarise': Predict},
            {'extract': Predict, 'classify': ChainOfThought,
             'reason': Predict, 'summarise': Predict},
            {'extract': Predict, 'classify': ChainOfThought,
             'reason': ChainOfThought, 'summarise': Predict},
            {'extract': Predict, 'classify': Predict,
             'reason': ChainOfThought, 'summarise': Predict},
            {'extract': ChainOfThought, 'classify': ChainOfThought,
             'reason': ChainOfThought, 'summarise': ChainOfThought},
        ]

        for pq in [0.3, 0.45, 0.6, 0.75]:
            for use_demos in [False, True]:
                for mi, mtypes in enumerate(module_combos):
                    d = demos if use_demos else None
                    pipe = self._build_pipeline(mtypes, pq, d)
                    df = evaluate_pipeline(pipe, self.val)
                    score = df['composite'].mean()

                    combo_label = '+'.join(
                        'CoT' if v == ChainOfThought else 'P'
                        for v in mtypes.values()
                    )

                    trial = {
                        'prompt_quality': pq,
                        'demos': use_demos,
                        'module_combo': combo_label,
                        'combo_idx': mi,
                        'composite': round(score, 4),
                        'entity_f1': round(df['entity_f1'].mean(), 4),
                        'classify_acc': round(df['classify_acc'].mean(), 4),
                        'reasoning_cov': round(df['reasoning_cov'].mean(), 4),
                        'summary_qual': round(df['summary_qual'].mean(), 4),
                    }
                    trials.append(trial)

                    if score > best_score:
                        best_score = score
                        best_config = {
                            'pq': pq, 'demos': use_demos,
                            'module_types': mtypes, 'demo_data': d,
                            'combo_label': combo_label,
                        }

        print(f'  Evaluated {len(trials)} configurations')
        print(f'  Best composite: {best_score:.4f}')
        print(f'  Best config: pq={best_config["pq"]}, '
              f'demos={best_config["demos"]}, '
              f'modules={best_config["combo_label"]}')

        return best_config, pd.DataFrame(trials)


# Split: 60% train, 40% val
np.random.seed(42)
shuffled = ARTICLES.copy()
np.random.shuffle(shuffled)
split = int(len(shuffled) * 0.6)
train_arts = shuffled[:split]
val_arts = shuffled[split:]

opt = BootstrapOptimiser(LLM, train_arts, val_arts)
best_config, trial_df = opt.optimise()

### Optimisation Landscape Analysis

In [ ]:
print(f'Total trials: {len(trial_df)}')
print(f'Composite range: {trial_df["composite"].min():.4f} to {trial_df["composite"].max():.4f}')

print('\nTop 10 configurations:')
top10 = trial_df.nlargest(10, 'composite')
print(top10[['prompt_quality', 'demos', 'module_combo', 'composite',
             'entity_f1', 'classify_acc']].to_string(index=False))

print('\nPrompt quality impact (averaged):')
for pq, grp in trial_df.groupby('prompt_quality'):
    print(f'  pq={pq:.2f}: composite={grp["composite"].mean():.4f} '
          f'(best={grp["composite"].max():.4f})')

print('\nDemonstration impact:')
for demos, grp in trial_df.groupby('demos'):
    print(f'  demos={str(demos):<6s}: composite={grp["composite"].mean():.4f}')

In [ ]:
fig = make_subplots(rows=1, cols=3,
                    subplot_titles=['By Prompt Quality', 'By Module Combo',
                                    'Demos vs No-Demos'])

for pq in sorted(trial_df['prompt_quality'].unique()):
    grp = trial_df[trial_df['prompt_quality'] == pq]
    fig.add_trace(go.Box(y=grp['composite'], name=f'pq={pq}'),
                  row=1, col=1)

for combo in trial_df['module_combo'].unique():
    grp = trial_df[trial_df['module_combo'] == combo]
    fig.add_trace(go.Box(y=grp['composite'], name=combo[:12],
                         showlegend=False), row=1, col=2)

for demos in [False, True]:
    grp = trial_df[trial_df['demos'] == demos]
    fig.add_trace(go.Box(y=grp['composite'],
                         name=f'demos={demos}', showlegend=False),
                  row=1, col=3)

fig.update_layout(height=420, template='plotly_white',
                  title_text='Optimisation Landscape: What Matters Most?',
                  showlegend=False)
fig.update_yaxes(range=[0, 1])
fig.show()

### Build the Optimised Pipeline

In [ ]:
pipe_optimised = MultiStepPipeline(
    steps=[
        ('extract', best_config['module_types']['extract'](
            SIG_EXTRACT, LLM, prompt_quality=best_config['pq'],
            demos=best_config.get('demo_data', {}).get('extract') if best_config['demos'] else None)),
        ('classify', best_config['module_types']['classify'](
            SIG_CLASSIFY, LLM, prompt_quality=best_config['pq'],
            demos=best_config.get('demo_data', {}).get('classify') if best_config['demos'] else None)),
        ('reason', best_config['module_types']['reason'](
            SIG_REASON, LLM, prompt_quality=best_config['pq'],
            demos=best_config.get('demo_data', {}).get('reason') if best_config['demos'] else None)),
        ('summarise', best_config['module_types']['summarise'](
            SIG_SUMMARISE, LLM, prompt_quality=best_config['pq'],
            demos=best_config.get('demo_data', {}).get('summarise') if best_config['demos'] else None)),
    ],
    name='DSPy-Optimised',
)

df_optimised = evaluate_pipeline(pipe_optimised, ARTICLES)

print('DSPY-OPTIMISED PIPELINE')
print('=' * 55)
for col in ['entity_f1', 'classify_acc', 'reasoning_cov', 'summary_qual', 'composite']:
    print(f'  {col:<16s}: {df_optimised[col].mean():.4f}')

In [ ]:
# Also build a 'best manual guess' for comparison
manual_demos = opt._bootstrap_demos(pq=0.5)
pipe_manual_best = MultiStepPipeline([
    ('extract', Predict(SIG_EXTRACT, LLM, prompt_quality=0.60,
                        demos=manual_demos['extract'])),
    ('classify', ChainOfThought(SIG_CLASSIFY, LLM, prompt_quality=0.60,
                                demos=manual_demos['classify'])),
    ('reason', ChainOfThought(SIG_REASON, LLM, prompt_quality=0.60,
                              demos=manual_demos['reason'])),
    ('summarise', Predict(SIG_SUMMARISE, LLM, prompt_quality=0.60,
                          demos=manual_demos['summarise'])),
], name='Manual Best Guess')

df_manual_best = evaluate_pipeline(pipe_manual_best, ARTICLES)

print('MANUAL BEST GUESS (CoT + demos, pq=0.60)')
print('=' * 55)
for col in ['entity_f1', 'classify_acc', 'reasoning_cov', 'summary_qual', 'composite']:
    print(f'  {col:<16s}: {df_manual_best[col].mean():.4f}')

## Part 7: Full Comparison

All five approaches evaluated on the complete dataset.

In [ ]:
ALL = {
    'Imperative Baseline': df_imperative,
    'Hand-Tuned': df_tuned,
    'Declarative + CoT': df_cot,
    'Manual Best Guess': df_manual_best,
    'DSPy-Optimised': df_optimised,
}

PIPE_COLORS = {
    'Imperative Baseline': '#e74c3c',
    'Hand-Tuned': '#e67e22',
    'Declarative + CoT': '#3498db',
    'Manual Best Guess': '#9b59b6',
    'DSPy-Optimised': '#2ecc71',
}

comp_rows = []
for name, df in ALL.items():
    comp_rows.append({
        'Pipeline': name,
        'Entity F1': round(df['entity_f1'].mean(), 4),
        'Classify Acc': round(df['classify_acc'].mean(), 4),
        'Reasoning Cov': round(df['reasoning_cov'].mean(), 4),
        'Summary Qual': round(df['summary_qual'].mean(), 4),
        'Composite': round(df['composite'].mean(), 4),
    })

comp_df = pd.DataFrame(comp_rows)
print('PIPELINE COMPARISON')
print('=' * 85)
print(comp_df.to_string(index=False))

# Lift over baseline
base = df_imperative['composite'].mean()
print(f'\nLift over Imperative Baseline (composite):')
for name, df in ALL.items():
    if name != 'Imperative Baseline':
        lift = df['composite'].mean() - base
        pct = lift / base * 100 if base > 0 else 0
        print(f'  {name:<22s}: +{lift:.4f} ({pct:+.1f}%)')

### Visualisations

In [ ]:
metrics_list = ['Entity F1', 'Classify Acc', 'Reasoning Cov', 'Summary Qual', 'Composite']

fig = make_subplots(rows=1, cols=5, subplot_titles=metrics_list)

for col_idx, metric in enumerate(metrics_list, 1):
    for _, row in comp_df.iterrows():
        fig.add_trace(go.Bar(
            x=[row['Pipeline'][:12]], y=[row[metric]],
            marker_color=PIPE_COLORS.get(row['Pipeline'], '#95a5a6'),
            name=row['Pipeline'], showlegend=(col_idx == 1),
        ), row=1, col=col_idx)

fig.update_layout(height=430, template='plotly_white',
                  title_text='Multi-Step Pipeline Comparison',
                  showlegend=False)
fig.update_yaxes(range=[0, 1])
fig.show()

In [ ]:
radar_dims = ['Entity F1', 'Classify Acc', 'Reasoning Cov', 'Summary Qual']

fig = go.Figure()
for _, row in comp_df.iterrows():
    vals = [row[d] for d in radar_dims]
    fig.add_trace(go.Scatterpolar(
        r=vals + [vals[0]],
        theta=radar_dims + [radar_dims[0]],
        name=row['Pipeline'],
        fill='toself',
        line_color=PIPE_COLORS.get(row['Pipeline'], '#95a5a6'),
        opacity=0.55,
    ))

fig.update_layout(
    polar=dict(radialaxis=dict(range=[0, 1], visible=True)),
    title='Per-Step Quality Radar',
    template='plotly_white', height=500,
)
fig.show()

In [ ]:
trajectory_data = [
    (name, comp_df.loc[comp_df['Pipeline'] == name, 'Composite'].values[0])
    for name in ['Imperative Baseline', 'Hand-Tuned', 'Declarative + CoT',
                 'Manual Best Guess', 'DSPy-Optimised']
]

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=list(range(len(trajectory_data))),
    y=[t[1] for t in trajectory_data],
    mode='lines', line=dict(color='#bdc3c7', width=2, dash='dot'),
    showlegend=False,
))
for i, (label, val) in enumerate(trajectory_data):
    fig.add_trace(go.Scatter(
        x=[i], y=[val], mode='markers+text',
        marker=dict(size=18, color=PIPE_COLORS[label],
                    line=dict(color='white', width=2)),
        text=[f'{val:.1%}'], textposition='top center',
        showlegend=False,
    ))

short_labels = ['Imperative\nBaseline', 'Hand\nTuned', 'Declarative\n+ CoT',
                'Manual\nBest', 'DSPy\nOptimised']
fig.update_layout(
    title='Composite Score Trajectory: From Imperative to Optimised',
    xaxis=dict(tickmode='array', tickvals=list(range(len(short_labels))),
               ticktext=short_labels),
    yaxis_title='Composite Score', yaxis_range=[0, 1],
    template='plotly_white', height=420,
)
fig.show()

In [ ]:
# Per-domain comparison
all_per_domain = []
for name, df in ALL.items():
    for _, row in df.iterrows():
        all_per_domain.append({
            'pipeline': name, 'domain': row['domain'],
            'composite': row['composite'],
        })

dom_df = pd.DataFrame(all_per_domain)

fig = px.box(dom_df, x='domain', y='composite', color='pipeline',
             color_discrete_map=PIPE_COLORS,
             title='Composite Score by Domain',
             template='plotly_white',
             labels={'composite': 'Composite Score', 'domain': 'Domain'})
fig.update_layout(height=450)
fig.show()

In [ ]:
# Heatmap: prompt quality vs module combo
hm_data = trial_df.groupby(['prompt_quality', 'module_combo'])['composite'].mean().unstack()

fig = px.imshow(
    hm_data.values,
    x=[c[:14] for c in hm_data.columns],
    y=[f'pq={r:.2f}' for r in hm_data.index],
    color_continuous_scale='YlOrRd',
    text_auto='.3f',
    title='Optimisation Heatmap: Prompt Quality vs Module Strategy',
    labels=dict(x='Module Combo', y='Prompt Quality', color='Composite'),
)
fig.update_layout(template='plotly_white', height=350)
fig.show()

In [ ]:
# Step-by-step quality flow: how does quality propagate through steps?
step_metrics = ['entity_f1', 'classify_acc', 'reasoning_cov', 'summary_qual']
step_labels = ['Extract', 'Classify', 'Reason', 'Summarise']

fig = go.Figure()
for name, df in ALL.items():
    vals = [df[m].mean() for m in step_metrics]
    fig.add_trace(go.Scatter(
        x=step_labels, y=vals,
        mode='lines+markers',
        name=name,
        line=dict(color=PIPE_COLORS[name], width=2.5),
        marker=dict(size=10),
    ))

fig.update_layout(
    title='Quality Propagation Through Pipeline Steps',
    xaxis_title='Pipeline Step', yaxis_title='Step Quality',
    yaxis_range=[0, 1], template='plotly_white', height=430,
)
fig.show()

### Error Analysis: Where Does Each Pipeline Fail?

In [ ]:
# Classification confusion analysis
print('CLASSIFICATION ERRORS BY PIPELINE')
print('=' * 55)
for name, df in ALL.items():
    n_wrong = (df['classify_acc'] == 0).sum()
    n_total = len(df)
    pct = n_wrong / n_total * 100
    print(f'  {name:<22s}: {n_wrong}/{n_total} wrong ({pct:.0f}%)')
    if n_wrong > 0:
        wrong = df[df['classify_acc'] == 0]
        for _, row in wrong.head(2).iterrows():
            art = next(a for a in ARTICLES if a['id'] == row['article_id'])
            print(f'    {row["article_id"]}: predicted={row["topic_pred"]}, '
                  f'expected={art["gt_topic"]}')

In [ ]:
# Per-step improvement waterfall
baseline_vals = [df_imperative[m].mean() for m in step_metrics]
optimised_vals = [df_optimised[m].mean() for m in step_metrics]
deltas = [o - b for o, b in zip(optimised_vals, baseline_vals)]

fig = go.Figure(go.Waterfall(
    x=step_labels,
    y=deltas,
    measure=['relative'] * 4,
    text=[f'+{d:.3f}' if d >= 0 else f'{d:.3f}' for d in deltas],
    increasing_marker_color='#2ecc71',
    decreasing_marker_color='#e74c3c',
    connector_line_color='#bdc3c7',
))
fig.update_layout(
    title='Improvement Waterfall: DSPy-Optimised vs Imperative Baseline',
    yaxis_title='Score Improvement', template='plotly_white', height=400,
)
fig.show()

## Key Lessons

### Lesson 1: Signatures Separate What from How

```
Imperative:  'Extract entities from the following text. Return them as a
              comma-separated list. Only include technical terms. Do not
              include common English words. The text is: {text}'

Declarative: text -> entities
```

The signature is **model-agnostic**. The same program works with GPT-4,
Llama 3, or Qwen3 -- only the optimiser's demonstrations change.

### Lesson 2: Multi-Step Errors Compound

If entity extraction has 80% accuracy and classification depends on entities,
the cascading error rate is worse than 20%. **End-to-end optimisation**
accounts for error propagation between steps.

### Lesson 3: Chain-of-Thought Is Step-Specific

CoT helps most on **reasoning-heavy steps** (classification, argument analysis)
but adds unnecessary cost to **extraction and summarisation** steps.
The optimiser discovers this automatically.

### Lesson 4: Demonstrations > Instructions

Showing the model 3-5 examples of good output is more reliable than
writing detailed instructions. BootstrapFewShot selects demonstrations
that maximise your metric on training data.

## Production Patterns

### Complete DSPy Multi-Step Workflow

```python
import dspy

# 1. Configure LLM
lm = dspy.LM('ollama_chat/qwen3', api_base='http://localhost:11434')
dspy.configure(lm=lm)

# 2. Define Signatures
class ExtractEntities(dspy.Signature):
    text: str = dspy.InputField(desc='technical article')
    entities: list[str] = dspy.OutputField(desc='key technical entities')

class ClassifyTopic(dspy.Signature):
    text: str = dspy.InputField()
    entities: list[str] = dspy.InputField()
    topic: str = dspy.OutputField(desc='one of: deep_learning, nlp, systems, ...')

class AnalyseArgument(dspy.Signature):
    text: str = dspy.InputField()
    entities: list[str] = dspy.InputField()
    topic: str = dspy.InputField()
    analysis: str = dspy.OutputField(desc='key argument and contribution')

class Summarise(dspy.Signature):
    analysis: str = dspy.InputField()
    summary: str = dspy.OutputField(desc='2-sentence summary')

# 3. Build the Module
class ArticleAnalyser(dspy.Module):
    def __init__(self):
        self.extract = dspy.Predict(ExtractEntities)
        self.classify = dspy.ChainOfThought(ClassifyTopic)
        self.reason = dspy.ChainOfThought(AnalyseArgument)
        self.summarise = dspy.Predict(Summarise)

    def forward(self, text):
        e = self.extract(text=text)
        c = self.classify(text=text, entities=e.entities)
        r = self.reason(text=text, entities=e.entities, topic=c.topic)
        s = self.summarise(analysis=r.analysis)
        return dspy.Prediction(
            entities=e.entities, topic=c.topic,
            analysis=r.analysis, summary=s.summary,
        )

# 4. Define Metric
def composite_metric(example, pred, trace=None):
    ef1 = entity_f1(pred.entities, example.entities)
    acc = 1.0 if pred.topic == example.topic else 0.0
    cov = reasoning_coverage(pred.analysis, example.entities)
    sq = summary_quality(pred.summary, pred.analysis)
    return (ef1 + acc + cov + sq) / 4

# 5. Optimise
optimizer = dspy.MIPROv2(metric=composite_metric, auto='medium')
optimised = optimizer.compile(
    ArticleAnalyser(),
    trainset=train_examples,
    valset=val_examples,
)

# 6. Save -> deploy
optimised.save('article_analyser_v1.json')
loaded = ArticleAnalyser()
loaded.load('article_analyser_v1.json')
```

### Adding Assertions for Reliability

```python
class ReliableAnalyser(dspy.Module):
    def __init__(self):
        self.extract = dspy.Predict(ExtractEntities)
        self.classify = dspy.ChainOfThought(ClassifyTopic)
        self.reason = dspy.ChainOfThought(AnalyseArgument)
        self.summarise = dspy.Predict(Summarise)

    def forward(self, text):
        e = self.extract(text=text)
        dspy.Assert(len(e.entities) >= 2, 'Must extract at least 2 entities')

        c = self.classify(text=text, entities=e.entities)
        dspy.Suggest(c.topic != 'unknown', 'Topic should be specific')

        r = self.reason(text=text, entities=e.entities, topic=c.topic)
        dspy.Assert(
            any(ent.lower() in r.analysis.lower() for ent in e.entities),
            'Analysis must reference at least one extracted entity',
        )

        s = self.summarise(analysis=r.analysis)
        return dspy.Prediction(
            entities=e.entities, topic=c.topic,
            analysis=r.analysis, summary=s.summary,
        )
```

**Assert** = hard constraint (retry on failure). **Suggest** = soft preference.

## Summary

### The Declarative Advantage

```
Imperative Prompt Chain           Declarative DSPy Program
─────────────────────             ────────────────────────
4 hand-written prompts       ->  4 typed Signatures
Fragile to LLM changes      ->  Model-agnostic
Manual few-shot selection    ->  BootstrapFewShot auto-selects
No end-to-end optimisation   ->  MIPROv2 optimises holistically
Hours per iteration          ->  Minutes of compute
Error propagation invisible  ->  Measured and optimised
```

### When to Use Each Approach

| Scenario | Recommendation |
|---|---|
| Quick demo / prototype | Imperative prompts are fine |
| Switching LLMs often | Declarative -- recompile, same code |
| Production multi-step pipeline | Declarative + optimisation |
| Debugging quality regressions | Measure each step independently |
| Team of prompt engineers | Declarative reduces coordination |

### Key Takeaways

| # | Insight |
|---|---|
| 1 | **Signatures separate concerns**: what vs how |
| 2 | **End-to-end optimisation** beats step-by-step tuning |
| 3 | **Error propagation** in multi-step pipelines is the main bottleneck |
| 4 | **Demonstrations** (bootstrapped) > instructions (hand-written) |
| 5 | **CoT is step-specific**: use it where reasoning helps, skip where it doesn't |
| 6 | **Assertions** catch runtime failures without manual QA |

---
*Educational notebook -- NLP / Declarative LLM Workflow Portfolio.*